In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
from sklearn.model_selection import GroupShuffleSplit
from Bio.Align import substitution_matrices
from collections import Counter
from treenode import TreeNode
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

import os
import json
import joblib
from datetime import datetime

from utils import save_model 

/home/hamdaalhosani/Downloads/yes/envs/benchmark/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load train / validation / test datasets
dataset_train = pd.read_csv("../data/dataset_train.csv")
dataset_val   = pd.read_csv("../data/dataset_val.csv")
dataset_test  = pd.read_csv("../data/dataset_test.csv")

# Drop non-feature columns
DROP_COLS = ["index", "peptide", "HLA", "hla_sequence"]
TARGET_COL = "Label"

# Split labels
y_train = dataset_train[TARGET_COL].values
y_val   = dataset_val[TARGET_COL].values
y_test  = dataset_test[TARGET_COL].values

# Split features
X_train = dataset_train.drop(columns=[TARGET_COL] + [c for c in DROP_COLS if c in dataset_train.columns])
X_val   = dataset_val.drop(columns=[TARGET_COL] + [c for c in DROP_COLS if c in dataset_val.columns])
X_test  = dataset_test.drop(columns=[TARGET_COL] + [c for c in DROP_COLS if c in dataset_test.columns])

# Combine before encoding so all splits have the same feature columns
X_all = pd.concat(
    [X_train, X_val, X_test],
    axis=0,
    keys=["train", "val", "test"]
)

# Identify categorical columns
cat_cols = X_all.select_dtypes(include=["object", "category"]).columns.tolist()
print(f"Encoding categorical columns: {cat_cols}")

# One-hot encode
X_all = pd.get_dummies(X_all, columns=cat_cols)

# Split back
X_train = X_all.loc["train"]
X_val   = X_all.loc["val"]
X_test  = X_all.loc["test"]

print(f"Train: {X_train.shape}, Classes: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Val:   {X_val.shape}, Classes: {dict(zip(*np.unique(y_val, return_counts=True)))}")
print(f"Test:  {X_test.shape}, Classes: {dict(zip(*np.unique(y_test, return_counts=True)))}")

print(X_train.columns.tolist())

Encoding categorical columns: []
Train: (5707, 290), Classes: {np.int64(0): np.int64(3143), np.int64(1): np.int64(2564)}
Val:   (1446, 290), Classes: {np.int64(0): np.int64(778), np.int64(1): np.int64(668)}
Test:  (1815, 290), Classes: {np.int64(0): np.int64(991), np.int64(1): np.int64(824)}
['PeptidePos_p1_f1', 'PeptidePos_p1_f2', 'PeptidePos_p1_f3', 'PeptidePos_p1_f4', 'PeptidePos_p1_f5', 'PeptidePos_p1_f6', 'PeptidePos_p1_f7', 'PeptidePos_p1_f8', 'PeptidePos_p1_f9', 'PeptidePos_p1_f10', 'PeptidePos_p1_f11', 'PeptidePos_p1_f12', 'PeptidePos_p1_f13', 'PeptidePos_p1_f14', 'PeptidePos_p1_f15', 'PeptidePos_p1_f16', 'PeptidePos_p1_f17', 'PeptidePos_p1_f18', 'PeptidePos_p2_f1', 'PeptidePos_p2_f2', 'PeptidePos_p2_f3', 'PeptidePos_p2_f4', 'PeptidePos_p2_f5', 'PeptidePos_p2_f6', 'PeptidePos_p2_f7', 'PeptidePos_p2_f8', 'PeptidePos_p2_f9', 'PeptidePos_p2_f10', 'PeptidePos_p2_f11', 'PeptidePos_p2_f12', 'PeptidePos_p2_f13', 'PeptidePos_p2_f14', 'PeptidePos_p2_f15', 'PeptidePos_p2_f16', 'PeptidePo

In [3]:
# Decision Tree implementation is based on the CART Algorithm

# Compute Gini impurity
def gini(y):
    if len(y) == 0:
        return 0.0
    counts = np.bincount(y)
    probs = counts / len(y)
    return 1.0 - np.sum(probs ** 2)


# Compute Gini-based information gain
def information_gain(y_parent, y_left, y_right):
    n = len(y_parent)
    weighted_child = (len(y_left) / n) * gini(y_left) + \
                     (len(y_right) / n) * gini(y_right)
    return gini(y_parent) - weighted_child


# Find the best feature and threshold to split on
def best_split(X, y):
    best_gain = -1
    best_feat, best_thresh = None, None

    for feat in range(X.shape[1]):
        thresholds = np.unique(X[:, feat])

        for thresh in thresholds:
            left_mask = X[:, feat] <= thresh
            right_mask = ~left_mask

            if left_mask.sum() == 0 or right_mask.sum() == 0:
                continue

            gain = information_gain(y, y[left_mask], y[right_mask])

            if gain > best_gain:
                best_gain = gain
                best_feat = feat
                best_thresh = thresh

    return best_feat, best_thresh, best_gain


# Create a leaf node
def make_leaf(y, n_samples):
    unique, counts = np.unique(y, return_counts=True)
    label_counts = dict(zip(unique.tolist(), counts.tolist()))
    probs = {k: v / n_samples for k, v in label_counts.items()}

    return TreeNode(
        n_samples=n_samples,
        prediction_probs=probs,
        label_counts=label_counts
    )


# Recursively build the decision tree
def build_tree(X, y, depth=0, max_depth=10, min_samples_split=2, min_samples_leaf=1):
    n_samples = len(y)

    if (
        depth >= max_depth
        or n_samples < min_samples_split
        or len(set(y)) == 1
    ):
        return make_leaf(y, n_samples)

    feat, thresh, gain = best_split(X, y)

    if feat is None or gain <= 0:
        return make_leaf(y, n_samples)

    left_mask = X[:, feat] <= thresh
    right_mask = ~left_mask

    if left_mask.sum() < min_samples_leaf or right_mask.sum() < min_samples_leaf:
        return make_leaf(y, n_samples)

    node = TreeNode(
        feature_idx=feat,
        feature_val=thresh,
        information_gain=gain,
        n_samples=n_samples
    )

    node.left = build_tree(
        X[left_mask], y[left_mask],
        depth + 1, max_depth, min_samples_split, min_samples_leaf
    )

    node.right = build_tree(
        X[right_mask], y[right_mask],
        depth + 1, max_depth, min_samples_split, min_samples_leaf
    )

    return node


# Predict label for a single sample
def predict_one(node, x):
    if node.is_leaf:
        return max(node.prediction_probs, key=node.prediction_probs.get)

    if x[node.feature_idx] <= node.feature_val:
        return predict_one(node.left, x)
    else:
        return predict_one(node.right, x)


# Predict labels for multiple samples
def predict(root, X):
    return np.array([predict_one(root, x) for x in X])


# Aggregate feature importance based on information gain
def get_feature_importances(node, n_features):
    importances = np.zeros(n_features)

    def walk(n):
        if n is None or n.is_leaf:
            return

        importances[n.feature_idx] += n.feature_importance

        walk(n.left)
        walk(n.right)

    walk(node)

    total = importances.sum()
    return importances / total if total > 0 else importances


# CART decision tree classifier implementation
class CARTClassifier:
    def __init__(self, max_depth=10, min_samples_split=2, min_samples_leaf=1):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.root = None
        self.feature_names_ = None

    def fit(self, X, y, feature_names=None):
        X = X.values if isinstance(X, pd.DataFrame) else X

        self.root = build_tree(
            X,
            y,
            max_depth=self.max_depth,
            min_samples_split=self.min_samples_split,
            min_samples_leaf=self.min_samples_leaf
        )

        self.feature_names_ = feature_names
        return self

    def predict(self, X):
        X = X.values if isinstance(X, pd.DataFrame) else X
        return predict(self.root, X)

    def predict_proba(self, X):
        X = X.values if isinstance(X, pd.DataFrame) else X

        probs = []

        for x in X:
            node = self.root

            while not node.is_leaf:
                if x[node.feature_idx] <= node.feature_val:
                    node = node.left
                else:
                    node = node.right

            probs.append([
                node.prediction_probs.get(0, 0),
                node.prediction_probs.get(1, 0)
            ])

        return np.array(probs)

    def score(self, X, y):
        return np.mean(self.predict(X) == y)

    def feature_importances(self):
        imps = get_feature_importances(self.root, len(self.feature_names_ or []))

        if self.feature_names_:
            return pd.Series(imps, index=self.feature_names_).sort_values(ascending=False)

        return imps


# Training and evaluating the model using the new train / val / test splits

cart = CARTClassifier(
    max_depth=6,
    min_samples_split=5,
    min_samples_leaf=2
)

cart.fit(
    X_train,
    y_train,
    feature_names=X_train.columns.tolist()
)

print(f"Train accuracy:      {cart.score(X_train, y_train):.4f}")
print(f"Validation accuracy: {cart.score(X_val, y_val):.4f}")
print(f"Test accuracy:       {cart.score(X_test, y_test):.4f}")

print("\nTop 10 most important features:")
print(cart.feature_importances().head(10))

Train accuracy:      0.7945
Validation accuracy: 0.7621
Test accuracy:       0.7708

Top 10 most important features:
HLA_F3               0.526968
HLA_VSTPV2           0.134031
HLA_BLOSUM6          0.101292
HLA_PD1              0.038498
PeptidePos_p4_f17    0.027465
HLA_F4               0.024108
PeptidePos_p6_f4     0.011101
HLA_AF3              0.009942
PeptidePos_p5_f7     0.009458
PeptidePos_p4_f4     0.008887
dtype: float64


In [4]:
# Validation
y_val_pred = cart.predict(X_val)
y_val_prob = cart.predict_proba(X_val)[:, 1]
print("-- Validation --")
print(f"Accuracy : {accuracy_score(y_val, y_val_pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_val, y_val_prob):.4f}")
print(f"AUPRC    : {average_precision_score(y_val, y_val_prob):.4f}")

# Test
y_test_pred = cart.predict(X_test)
y_test_prob = cart.predict_proba(X_test)[:, 1]
print("\n-- Test --")
print(f"Accuracy : {accuracy_score(y_test, y_test_pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_test_prob):.4f}")
print(f"AUPRC    : {average_precision_score(y_test, y_test_prob):.4f}")
print(classification_report(y_test, y_test_pred))

-- Validation --
Accuracy : 0.7621
ROC-AUC  : 0.8176
AUPRC    : 0.7499

-- Test --
Accuracy : 0.7708
ROC-AUC  : 0.8311
AUPRC    : 0.7610
              precision    recall  f1-score   support

           0       0.83      0.73      0.78       991
           1       0.72      0.82      0.77       824

    accuracy                           0.77      1815
   macro avg       0.77      0.78      0.77      1815
weighted avg       0.78      0.77      0.77      1815



In [5]:
test_auprc = average_precision_score(y_test, y_test_prob)
print("Test AUPRC:", round(test_auprc, 4))

Test AUPRC: 0.761


In [6]:
OUTPUT_DIR = "../models"
MODEL_NAME = "decision_tree_scratch_v1"

metadata = {
    "dataset": "dataset_train / dataset_val / dataset_test",
    "features_shape": X_train.shape,
    "features": X_train.columns.tolist(),
    "max_depth": cart.max_depth,
    "min_samples_split": cart.min_samples_split,
    "min_samples_leaf": cart.min_samples_leaf,
    "val_metrics": {
        "accuracy": accuracy_score(y_val, y_val_pred),
        "roc_auc": roc_auc_score(y_val, y_val_prob),
        "auprc": average_precision_score(y_val, y_val_prob),
    },
    "test_metrics": {
        "accuracy": accuracy_score(y_test, y_test_pred),
        "roc_auc": roc_auc_score(y_test, y_test_prob),
        "auprc": average_precision_score(y_test, y_test_prob),
    },
    "notes": "Custom CART decision tree implementation"
}

model_path = save_model(
    model=cart,
    output_dir=OUTPUT_DIR,
    model_name=MODEL_NAME,
    metadata=metadata
)

Saved model: ../models/decision_tree_scratch_v1.pkl
Saved metadata: ../models/decision_tree_scratch_v1_metadata.json


In [7]:
# random search for CART Decision Tree

configs = [
    {"max_depth": 3,  "min_samples_split": 2,  "min_samples_leaf": 1},
    {"max_depth": 5,  "min_samples_split": 2,  "min_samples_leaf": 1},
    {"max_depth": 6,  "min_samples_split": 5,  "min_samples_leaf": 2},
    {"max_depth": 8,  "min_samples_split": 5,  "min_samples_leaf": 2},
    {"max_depth": 10, "min_samples_split": 10, "min_samples_leaf": 3},
    {"max_depth": 12, "min_samples_split": 10, "min_samples_leaf": 3},
]

results = []
best_model = None
best_score = -1
best_config = None

for i, cfg in enumerate(configs, start=1):
    version = f"v{i}"

    print(f"\nTraining decision_tree_scratch_{version}: {cfg}")

    cart = CARTClassifier(
        max_depth=cfg["max_depth"],
        min_samples_split=cfg["min_samples_split"],
        min_samples_leaf=cfg["min_samples_leaf"]
    )

    cart.fit(X_train, y_train, feature_names=X_train.columns.tolist())

    y_val_pred = cart.predict(X_val)
    y_val_prob = cart.predict_proba(X_val)[:, 1]

    val_accuracy = accuracy_score(y_val, y_val_pred)
    val_auc = roc_auc_score(y_val, y_val_prob)
    val_pr_auc = average_precision_score(y_val, y_val_prob)
    val_f1 = f1_score(y_val, y_val_pred)

    print(f"Accuracy: {val_accuracy:.4f}")
    print(f"ROC AUC:  {val_auc:.4f}")
    print(f"PR AUC:   {val_pr_auc:.4f}")
    print(f"F1:       {val_f1:.4f}")

    # Save model
    save_model(
        model=cart,
        output_dir="../models",
        model_name=f"decision_tree_scratch_{version}",
        metadata={
            "dataset": "dataset_train / dataset_val / dataset_test",
            "features": X_train.columns.tolist(),
            "config": cfg,
            "val_accuracy": val_accuracy,
            "val_auc": val_auc,
            "val_pr_auc": val_pr_auc,
            "val_f1": val_f1,
            "notes": "CART decision tree hyperparameter search model"
        }
    )

    results.append({
        "version": version,
        **cfg,
        "val_accuracy": val_accuracy,
        "val_auc": val_auc,
        "val_pr_auc": val_pr_auc,
        "val_f1": val_f1
    })

    # Select best based on AUC
    if val_auc > best_score:
        best_score = val_auc
        best_model = cart
        best_config = cfg

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("val_auc", ascending=False)

display(results_df)

print("\nBest Decision Tree config:")
print(best_config)
print(f"Best validation AUC: {best_score:.4f}")


Training decision_tree_scratch_v1: {'max_depth': 3, 'min_samples_split': 2, 'min_samples_leaf': 1}
Accuracy: 0.7427
ROC AUC:  0.7982
PR AUC:   0.7156
F1:       0.7486
Saved model: ../models/decision_tree_scratch_v1.pkl
Saved metadata: ../models/decision_tree_scratch_v1_metadata.json

Training decision_tree_scratch_v2: {'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 1}
Accuracy: 0.7586
ROC AUC:  0.8175
PR AUC:   0.7524
F1:       0.7554
Saved model: ../models/decision_tree_scratch_v2.pkl
Saved metadata: ../models/decision_tree_scratch_v2_metadata.json

Training decision_tree_scratch_v3: {'max_depth': 6, 'min_samples_split': 5, 'min_samples_leaf': 2}
Accuracy: 0.7621
ROC AUC:  0.8176
PR AUC:   0.7499
F1:       0.7543
Saved model: ../models/decision_tree_scratch_v3.pkl
Saved metadata: ../models/decision_tree_scratch_v3_metadata.json

Training decision_tree_scratch_v4: {'max_depth': 8, 'min_samples_split': 5, 'min_samples_leaf': 2}
Accuracy: 0.7642
ROC AUC:  0.8153
PR AUC:   0

,version,max_depth,min_samples_split,min_samples_leaf,val_accuracy,val_auc,val_pr_auc,val_f1
2,v3,6,5,2,0.762102,0.817649,0.749941,0.754286
1,v2,5,2,1,0.758645,0.817474,0.752379,0.755431
3,v4,8,5,2,0.764177,0.815333,0.756280,0.749449
4,v5,10,10,3,0.754495,0.813856,0.753787,0.734480
0,v1,3,2,1,0.742739,0.798192,0.715593,0.748649
5,v6,12,10,3,0.743430,0.792294,0.736071,0.718726



Best Decision Tree config:
{'max_depth': 6, 'min_samples_split': 5, 'min_samples_leaf': 2}
Best validation AUC: 0.8176
